In [0]:

# ============================================================
# CELDA 1. PREPARAR DATOS PARA XGBOOST SIN CLIMA
# ============================================================

# Nuestro objetivo es predecir si un vuelo tendrá
# un retraso de salida de 15 minutos o más.
#
# Utilizaremos información programada del vuelo.
# No incluiremos clima ni retrasos reales como predictores.
#
# Los dos modelos utilizarán la misma muestra y
# los mismos periodos para permitir una comparación justa.

from pyspark.sql import functions as F

TABLA_GOLD = "workspace.gold_flights.flights_weather_features"

vuelos = spark.table(TABLA_GOLD)

# ------------------------------------------------------------
# 1. SELECCIONAR LAS VARIABLES DEL MODELO
# ------------------------------------------------------------

variables = [
    "flight_date",
    "origin",
    "dest",
    "airline",
    "month",
    "day_of_week",
    "departure_hour",
    "is_weekend",
    "is_summer",
    "is_peak_hour",
    "delayed_15"
]

datos = vuelos.select(*variables)

# ------------------------------------------------------------
# 2. CREAR UNA MUESTRA REPRODUCIBLE
# ------------------------------------------------------------

# Usamos una selección determinista del 3% de los vuelos.
# No alteramos la proporción original de retrasos.
#
# La misma regla podrá utilizarla Diego para su modelo
# con clima, garantizando los mismos registros.

datos = datos.filter(
    F.pmod(
        F.xxhash64(
            "flight_date",
            "origin",
            "dest",
            "airline",
            "departure_hour"
        ),
        F.lit(1000)
    ) < 30
)

# ------------------------------------------------------------
# 3. DIVISIÓN CRONOLÓGICA
# ------------------------------------------------------------

# Entrenamiento: febrero 2024 a mayo 2025
# Validación: junio a agosto 2025
# Prueba: septiembre a diciembre 2025

train = datos.filter(
    F.col("flight_date") < F.lit("2025-06-01")
)

validation = datos.filter(
    (F.col("flight_date") >= F.lit("2025-06-01")) &
    (F.col("flight_date") < F.lit("2025-09-01"))
)

test = datos.filter(
    F.col("flight_date") >= F.lit("2025-09-01")
)

# ------------------------------------------------------------
# 4. VERIFICAR TAMAÑOS Y BALANCE DE CLASES
# ------------------------------------------------------------

print("=" * 55)
print("PREPARACIÓN DEL BASELINE XGBOOST")
print("=" * 55)

for nombre, conjunto in [
    ("Entrenamiento", train),
    ("Validación", validation),
    ("Prueba", test)
]:
    resumen = conjunto.agg(
        F.count("*").alias("total"),
        F.round(
            F.avg("delayed_15") * 100,
            2
        ).alias("delay_rate")
    ).first()

    print(
        f"{nombre}: "
        f"{resumen['total']:,} vuelos | "
        f"{resumen['delay_rate']}% retrasados"
    )

print("\nVariables predictoras:")
for variable in variables:
    if variable not in ["flight_date", "delayed_15"]:
        print("-", variable)

print("\nDATOS PREPARADOS CORRECTAMENTE.")


In [0]:

%pip install xgboost scikit-learn scipy


In [0]:

dbutils.library.restartPython()


In [0]:

# ============================================================
# CELDA 2. ENTRENAMIENTO DEL BASELINE XGBOOST SIN CLIMA
# ============================================================

# Objetivo:
# Predecir si un vuelo tendrá un retraso de salida
# de 15 minutos o más.
#
# No utilizamos información meteorológica.
# Tampoco utilizamos el retraso real como predictor.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# ------------------------------------------------------------
# 1. COMPROBAR QUE XGBOOST ESTÉ DISPONIBLE
# ------------------------------------------------------------

try:
    from xgboost import XGBClassifier
    import xgboost
    print("XGBoost disponible:", xgboost.__version__)
except ImportError:
    raise ImportError(
        "XGBoost no está instalado. "
        "Instala el paquete xgboost antes de continuar."
    )

# ------------------------------------------------------------
# 2. CONVERTIR LA MUESTRA A PANDAS
# ------------------------------------------------------------

# Los millones de registros permanecen en Spark.
# Solo convertimos la muestra preparada anteriormente.

train_pd = train.toPandas()
val_pd = validation.toPandas()
test_pd = test.toPandas()

print("Entrenamiento:", len(train_pd))
print("Validación:", len(val_pd))
print("Prueba:", len(test_pd))

# ------------------------------------------------------------
# 3. DEFINIR LAS VARIABLES PREDICTORAS
# ------------------------------------------------------------

categoricas = [
    "origin",
    "dest",
    "airline"
]

numericas = [
    "month",
    "day_of_week",
    "departure_hour",
    "is_weekend",
    "is_summer",
    "is_peak_hour"
]

objetivo = "delayed_15"

# Separamos las características del resultado real.
X_train = train_pd[categoricas + numericas]
X_val = val_pd[categoricas + numericas]
X_test = test_pd[categoricas + numericas]

y_train = train_pd[objetivo].astype(int)
y_val = val_pd[objetivo].astype(int)
y_test = test_pd[objetivo].astype(int)

# ------------------------------------------------------------
# 4. CODIFICAR LAS VARIABLES CATEGÓRICAS
# ------------------------------------------------------------

# XGBoost necesita una representación numérica.
# OneHotEncoder transforma aeropuertos y aerolíneas.
#
# El codificador aprende únicamente del entrenamiento.
# Los conjuntos de validación y prueba no se utilizan
# para aprender las categorías.

try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=True
    )

encoder.fit(X_train[categoricas])

from scipy.sparse import hstack, csr_matrix

def preparar_variables(df):
    categorias = encoder.transform(df[categoricas])

    numeros = csr_matrix(
        df[numericas]
        .astype("float32")
        .to_numpy()
    )

    return hstack(
        [categorias, numeros],
        format="csr",
        dtype=np.float32
    )

X_train_final = preparar_variables(X_train)
X_val_final = preparar_variables(X_val)
X_test_final = preparar_variables(X_test)

print("Variables codificadas:", X_train_final.shape[1])

# ------------------------------------------------------------
# 5. ENTRENAR XGBOOST
# ------------------------------------------------------------

modelo_baseline = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=2
)

modelo_baseline.fit(
    X_train_final,
    y_train,
    eval_set=[
        (X_val_final, y_val)
    ],
    verbose=False
)

print("MODELO XGBOOST ENTRENADO CORRECTAMENTE")

# ------------------------------------------------------------
# 6. EVALUAR VALIDACIÓN Y PRUEBA
# ------------------------------------------------------------

def evaluar_modelo(nombre, X, y):
    probabilidades = modelo_baseline.predict_proba(X)[:, 1]

    # Umbral inicial: 50%.
    predicciones = (probabilidades >= 0.5).astype(int)

    resultado = {
        "Conjunto": nombre,
        "Accuracy": accuracy_score(y, predicciones),
        "Precision": precision_score(
            y, predicciones, zero_division=0
        ),
        "Recall": recall_score(
            y, predicciones, zero_division=0
        ),
        "F1": f1_score(
            y, predicciones, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y, probabilidades)
    }

    return resultado, predicciones

metricas_val, pred_val = evaluar_modelo(
    "Validación", X_val_final, y_val
)

metricas_test, pred_test = evaluar_modelo(
    "Prueba", X_test_final, y_test
)

resultados_baseline = pd.DataFrame([
    metricas_val,
    metricas_test
])

print("\nMÉTRICAS DEL XGBOOST SIN CLIMA")
display(resultados_baseline.round(4))

# ------------------------------------------------------------
# 7. MATRIZ DE CONFUSIÓN DEL CONJUNTO DE PRUEBA
# ------------------------------------------------------------

matriz = confusion_matrix(y_test, pred_test)

ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=[
        "Sin retraso",
        "Retraso >=15 min"
    ]
).plot(values_format="d", cmap="Blues")

plt.title("XGBoost sin clima - Matriz de confusión")
plt.tight_layout()
plt.show()

print("\nBASELINE XGBOOST COMPLETADO.")


In [0]:

# ============================================================
# CELDA 3. AJUSTAR EL UMBRAL DEL XGBOOST
# ============================================================

# El modelo ya está entrenado.
# Probamos distintos umbrales usando VALIDACIÓN.
# El conjunto de PRUEBA solo se utiliza al final.

import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    confusion_matrix
)

# Probabilidades estimadas por el modelo
proba_val = modelo_baseline.predict_proba(X_val_final)[:, 1]
proba_test = modelo_baseline.predict_proba(X_test_final)[:, 1]

# Evaluar diferentes umbrales solo en validación
resultados_umbrales = []

for umbral in np.arange(0.10, 0.51, 0.025):
    pred = (proba_val >= umbral).astype(int)

    resultados_umbrales.append({
        "umbral": round(float(umbral), 3),
        "precision": precision_score(
            y_val, pred, zero_division=0
        ),
        "recall": recall_score(
            y_val, pred, zero_division=0
        ),
        "f1": f1_score(
            y_val, pred, zero_division=0
        )
    })

tabla_umbrales = pd.DataFrame(resultados_umbrales)

print("RESULTADOS POR UMBRAL EN VALIDACIÓN")
display(
    tabla_umbrales.sort_values(
        "f1", ascending=False
    ).head(10).round(4)
)

# Elegir el umbral con mayor F1 en validación
mejor_fila = tabla_umbrales.loc[
    tabla_umbrales["f1"].idxmax()
]

mejor_umbral = float(mejor_fila["umbral"])

print("\nMejor umbral según validación:", mejor_umbral)

# Aplicar el umbral seleccionado al conjunto de prueba
pred_test_ajustada = (
    proba_test >= mejor_umbral
).astype(int)

metricas_ajustadas = pd.DataFrame([{
    "Modelo": "XGBoost sin clima",
    "Umbral": mejor_umbral,
    "Accuracy": accuracy_score(
        y_test, pred_test_ajustada
    ),
    "Precision": precision_score(
        y_test, pred_test_ajustada, zero_division=0
    ),
    "Recall": recall_score(
        y_test, pred_test_ajustada, zero_division=0
    ),
    "F1": f1_score(
        y_test, pred_test_ajustada, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_test, proba_test
    )
}])

print("\nMÉTRICAS FINALES EN PRUEBA")
display(metricas_ajustadas.round(4))

print("\nMATRIZ DE CONFUSIÓN CON UMBRAL AJUSTADO")
print(confusion_matrix(y_test, pred_test_ajustada))

print("\nEVALUACIÓN DE UMBRAL COMPLETADA.")


In [0]:

# ============================================================
# CELDA 4. GUARDAR MÉTRICAS DEL BASELINE
# ============================================================

# Guardamos las métricas finales del modelo sin clima.
# El umbral fue seleccionado usando validación.
# Estas métricas corresponden al conjunto de prueba.

metricas_guardar = metricas_ajustadas.copy()

metricas_guardar.columns = [
    "model",
    "threshold",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc"
]

df_metricas_spark = spark.createDataFrame(
    metricas_guardar
)

(
    df_metricas_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.gold_flights.baseline_model_metrics"
    )
)

print("MÉTRICAS DEL BASELINE GUARDADAS CORRECTAMENTE")

display(
    spark.table(
        "workspace.gold_flights.baseline_model_metrics"
    )
)
